Model Fitting and Tuning: 6 points (out of 30).

An effective model is selected that is suitable for
the problem at hand, and a compelling case for that model is made. The model is properly
fit and tuned. A sound comparison with a baseline model is provided. Code is correct, model
is well justified, and can be passed to the client with no editing required

# 3 Model Fitting and Tuning

*In this section you should detail and motivate your choice of model and describe the process used to refine, tune, and fit that model. You are encouraged to explore different models but you should NOT include a detailed narrative or code of all of these attempts. At most this section should very briefly mention the methods explored and why they were rejected - most of your effort should go into describing the final model you are using and your process for tuning and validating it.*

*This section should include the full implementation of your final model, including all necessary validation. As with figures, any included code must also be addressed in the text of the document.*

*You should also include a baseline model of your choice and provide a comparison of your model with the baseline model on the test data. You should briefly describe the baseline model considered.*

## Model Fitting and Tuning **Mark Scheme**:

6 points (out of 30 for report). An effective model is selected that is suitable for
the problem at hand, and a compelling case for that model is made. The model is properly
fit and tuned. A sound comparison with a baseline model is provided. Code is correct, model
is well justified, and can be passed to the client with no editing required.

########################################

For the purposes of this classification, we considered classification trees, bagging, random forests, support vector machines, and logisitic regression. These frameworks were implemented with adequate tuning to produce a predictive model. The support vector machine approach failed to fit in a reasonable amount of runtime, as after over 45 minutes the pipeline still failed to produce results. For this reason, the support vector machine approach was removed from consideration for being impractical for the application and audience at hand. 

The primary metrics by which to assess a predictive model are specificity, recall, accuracy, and F1 Score. Of these metrics, the most pertinent is recall, or the rate at which the model correctly identifies children exhibiting depression as being depressed. In machine learning terms, this is the number of true positives (children with depression predicted to be depressed) divided by the sum of true positives and false negatives (children with depression predicted not to be depressed). We choose this metric as most important because failure to identify depression in children with depression, i.e. false negatives, lead to the most significant negative public health outcomes. That said, a model can achieve perfect recall by predicting positives for all data points, or in this case predicting every child is depressed. Therefore, while recall is the primary metric by which to assess model suitability, other performance metrics are also considered.

| Model                | Recall | Specificity | F1 Score | Accuracy |
|---------------------|--------|-------------|----------|----------|
| Decision Tree       | 0.91   | 0.13        | 0.34     | 0.29     |
| Bagging             | 0.05   | 0.97        | 0.08     | 0.79     |
| Logistic Regression | 0.58   | 0.60        | 0.37     | 0.59     |
| Random Forest       | 0.55   | 0.65        | 0.37     | 0.63     |

In this case, the approach with the best recall is the Classification Tree. However, the classification tree performs quite poorly across other performance metrics, and indeed achieved this high recall by dramatically overpredicting depression. The approach that achieves the highest recall balanced by reasonable performance across other metrics is Logistic Regression. This follows from the fact that logistic regression can be tuned specifically to perform better according to specific performance metrics, and thus could be tuned specifically to maximise recall. Logistic regression was therefore chosen as the approach from which to further develop a predictive model.

## 3.1 Logistic Regression

For the sake of model interpretation, we first define a function that will return the feature coefficients of the logistic regression and plot them. This allows for interpretation of which features are significant and which have little predictive power by looking at the magnitude of the associated coefficients. Features with high impact on the prediction will have coefficients significantly less than or greater than zero, while features with low impact on the prediction will have coefficients near zero.

In [ ]:
def get_coefs(m, plot = False, feature_names = None, figsize = (5,5), figtitle = None, intercept = True, top_k = None, min_abs_coeff = None, sort_by_abs = True):
    """Returns model coefficients in a data frame for a fitted linear model.
    
    Args:
        m: sklearn linear model object or pipeline with linear model as final step
        plot: boolean value, should coefficients be plotted with error bars
        feature_names: list of feature names to use in the plot 
        figsize: tuple defining figure size
        figtitle: string defining figure title
        intercept: boolean value, should intercept be included in the plot
    """
    
    # Extract intercept and coefficients into a single array
    w0 = m[-1].intercept_ if isinstance(m, sklearn.pipeline.Pipeline) else m.intercept_
    w1 = m[-1].coef_ if isinstance(m, sklearn.pipeline.Pipeline) else m.coef_
    w = np.concatenate((w0.reshape(-1), w1.reshape(-1)))
    # Extract name of features
    if feature_names is None:
        feature_names = m[:-1].get_feature_names_out() if isinstance(m, sklearn.pipeline.Pipeline) else m.feature_names_in_
    feature_names = np.concatenate((['intercept'], feature_names))
    # Create a data frame
    w_df = pd.DataFrame({'feature': feature_names, 'coef': w}).sort_values ("coef", ascending=False)
	    # get absolute value of coeffs
    w_df["abs_coef"] = w_df["coef"].abs()
    
    if sort_by_abs:
        w_df = w_df.sort_values("abs_coef", ascending=False)

    if min_abs_coeff is not None:
        w_df = w_df[w_df["abs_coef"] >= min_abs_coeff]

    if top_k is not None:
        w_df = w_df.head(top_k)
    if plot:
        if not intercept:
            w_df = w_df[w_df['feature'] != 'intercept']
        plt.figure(figsize=figsize)
        plt.barh(w_df['feature'], w_df['coef'])
        plt.ylabel('Features')
        plt.xlabel('Coefficient Value')
        plt.axvline(x=0, color=".5")
        if figtitle is not None:
            plt.title(figtitle)
        plt.grid()
        plt.show()
    
    return  w_df.drop(columns="abs_coef")


We now start by creating a logistic regression pipeline. Features are first standardized to have a mean of zero and variance of one, enabling the regression to assign coefficients/importance to features equivalently. Without this step, features with high mean or variance are punished by the regression and considered relatively unimportant because small changes in coefficient values associated with these features yield greater change in predictions compared to features with smaller mean or variance. Because of the imbalance in the data, we then assign entries with no reported depression a weight of 1 and entries with reported depression a weight of 7.86. This weighting is selected as there are approximately 7.86 times as many entries with no associated depression as there are with associated depression, therefore this weighting causes the regression to predict not based on the overall frequency of depression across the dataset, but instead on the case-by-case, observational basis desired. Because of the rigor of our exploratory data analysis, we use ridge regularization, or a penalty parameter of l2, to create a regression with coefficients balanced across all features. This tends coefficients toward zero without forcing them equal zero, giving predictions based on all features retained from the exploratory data analysis with the scale of coefficients still representing feature importance. The random seed is then set at 42 for reproducibility of results, and the solver and max iterations are set to balance quick convergence to a model with the quality obtained by that model.

Having chosen to obtain predictive coefficients by ridge regression, we must now tune the strength of the penalty parameter. We will do this by splitting the data into 5 stratified folds, or five sets with equal balance between positives (children exhibiting depression) and negatives (children exhibiting no depression). We then test different strengths of the penalty parameter by fitting the pipeline with across a set of strengths, fitting according to the best recall achieved at that strength, and plotting the recall achieved at all five sets and the average recall across all sets to determine which penalty parameter best balances predictive performance with model complexity.

In [ ]:
# Pipeline 
log_pipe_l2 = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight={0:1, 1:7.86}, random_state=42, penalty='l2', solver='liblinear',
        max_iter=1000)
)

# Possible C values: 
C_list = np.linspace(0,5, num=50)

# Grid search CV:
log_rs = GridSearchCV(log_pipe_l2, 
                      param_grid={'logisticregression__C': C_list},
                      scoring = ["accuracy", "f1","recall","precision"], #Evaluation metrics to compute on validation sets
                      cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                      refit = "recall", # Refits the best model on the entire dataset using the accuracy metric 
                      return_train_score = True)

# Tune the model with grid search:
log_rs.fit(X_train, y_train)

# Extract only mean and split scores
cv_accuracy = pd.DataFrame(
    data = log_rs.cv_results_
).filter(
    # Extract the split#_test_f1 and mean_test_f1 columns
    regex = '(split[0-4]+|mean)_test_recall'
).assign(
    # Add the alphas as a column
    C = C_list
)
# Reshape the data frame for plotting
d = cv_accuracy.melt(
    id_vars=('C','mean_test_recall'),
    var_name='fold',
    value_name='recall'
)
# Plot the validation scores across folds
plt.figure(figsize=(5,3))
sns.lineplot(x='C', y='recall', color='black', errorbar=None, data = d)  # Plot the mean score in black.
sns.lineplot(x='C', y='recall', hue='fold', data = d) # Plot the curves for each fold in different colors
plt.show()

From this we see that an appropriate penalty strength parameter is one. Using this parameter, we can fit the regression and obtain the following significant features and corresponding coefficients. 

In [ ]:
log_pipe_l2.fit(X_train, y_train)
get_coefs(log_pipe_l2, plot=True, min_abs_coeff=0.2)

Having fit the regression, we can assess its predictive performance on the testing data with the following confusion matrix and performance metrics of Accuracy, Precision, Recall, Specificity, and F1 Score.

In [ ]:
y_pred = log_pipe_l2.predict(X_test)
ConfusionMatrixDisplay.from_estimator(log_pipe_l2, X_test, y_test)
plt.show()
print(f"Accuracy: {metrics.accuracy_score(y_test, y_pred)}")
print(f"Precision: {metrics.precision_score(y_test, y_pred)}")
print(f"Sensitivity/Recall: {metrics.recall_score(y_test, y_pred)}")
print(f"Specificity: {metrics.recall_score(y_test, y_pred, pos_label=0.0)}")
print(f"F Score: {metrics.f1_score(y_test, y_pred)}")


Another way of dealing with the imbalance in data is to assign the weights automatically through the regressions built-in "balanced" class weights feature. This sacrifices interpretability of the class weights, but is worth investigating for the potential improvement in performance. We repeat the process as before, but with an updated, balanced class weight.

In [ ]:
log_pipe_bal = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", random_state=42, penalty='l2', solver='liblinear',
        max_iter=1000)
)

# Grid search CV:
log_rs = GridSearchCV(log_pipe_bal, 
                      param_grid={'logisticregression__C': C_list},
                      scoring = ["accuracy", "f1","recall","precision"], #Evaluation metrics to compute on validation sets
                      cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                      refit = "recall", # Refits the best model on the entire dataset using the accuracy metric 
                      return_train_score = True)

# Tune the model with grid search:
log_rs.fit(X_train, y_train)

# Extract only mean and split scores
cv_accuracy = pd.DataFrame(
    data = log_rs.cv_results_
).filter(
    # Extract the split#_test_f1 and mean_test_f1 columns
    regex = '(split[0-4]+|mean)_test_recall'
).assign(
    # Add the alphas as a column
    C = C_list
)
# Reshape the data frame for plotting
d = cv_accuracy.melt(
    id_vars=('C','mean_test_recall'),
    var_name='fold',
    value_name='recall'
)
# Plot the validation scores across folds
plt.figure(figsize=(5,3))
sns.lineplot(x='C', y='recall', color='black', errorbar=None, data = d)  # Plot the mean score in black.
sns.lineplot(x='C', y='recall', hue='fold', data = d) # Plot the curves for each fold in different colors
plt.show()

In this case, an appropriate penalty strength is 0.5. This can be used to fit the regression and obtain the following significant features and corresponding coefficients. 

In [ ]:
log_pipe_bal = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", random_state=42, penalty='l2', solver='liblinear',
        max_iter=1000, C=0.5)
)

log_pipe_bal.fit(X_train, y_train)
get_coefs(log_pipe_bal, plot=True, min_abs_coeff=0.2)

We can then again assess the predictive performance on the testing data as before by obtaining the following confusion matrix and performance metrics of Accuracy, Precision, Recall, Specificity, and F1 Score.

In [ ]:
y_pred = log_pipe_bal.predict(X_test)
ConfusionMatrixDisplay.from_estimator(log_pipe_bal, X_test, y_test)
plt.show()

print(f"Accuracy: {metrics.accuracy_score(y_test, y_pred)}")
print(f"Precision: {metrics.precision_score(y_test, y_pred)}")
print(f"Sensitivity/Recall: {metrics.recall_score(y_test, y_pred)}")
print(f"Specificity: {metrics.recall_score(y_test, y_pred, pos_label=0.0)}")
print(f"F Score: {metrics.f1_score(y_test, y_pred)}")